Ten notebook jest oceniany półautomatycznie. Nie twórz ani nie usuwaj komórek - struktura notebooka musi zostać zachowana. Odpowiedź wypełnij tam gdzie jest na to wskazane miejsce - odpowiedzi w innych miejscach nie będą sprawdzane (nie są widoczne dla sprawdzającego w systemie).

W szczególności zwróć uwagę, że usupełniłeś wszystkie miejsca `YOUR CODE HERE`, `WPISZ TWÓJ KOD TUTAJ`, "YOUR ANSWER HERE" lub "WPISZ TWOJĄ ODPOWIEDŹ TUTAJ".

### Zaawansowane Przetwarzanie Języka Naturalnego
# Laboratorium 2

Pobierz zbiór danych Amazon "Musical Instruments" z [tej](http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Musical_Instruments_5.json.gz) strony internetowej, a następnie wczytaj go poniższym kodem. Zwróć uwagę na wymaganą lokalizację pliku, tj. dwa katalogi wyżej - wynika to ze struktury plików w sprawdzarce, przepraszam za niedogodność.



In [ ]:
from collections import defaultdict, Counter
import time
import random
import torch
import json
 
x_text = []
y = []
# with open('Musical_Instruments_5.json') as f: # current repo structure
with open('../../Musical_Instruments_5.json') as f: # teacher's repo structure
    for line in f:
        data = json.loads(line)
        x_text.append(data['reviewText'].lower().strip())
        y.append(int(data['overall']))

## Zadanie 1 - przygotowanie danych
W załadowanych listach `x_text` oraz `y` znajdują się odpowiednio teksty kolejnych opinii oraz oznaczenia klas. Klasą w tym wypadku jest liczba gwiazdek (ocena) produktu towarzysząca opinii. Zadanie klasyfikacji polega na przewidzeniu oceny na podstawie opinii pozostawionej w portalu.

Aby zmniejszyć wymagania obliczeniowe do dalszych eksperymentów, ograniczymy zbiór danych jedynie do pierwszego tysiąca opinii.



In [2]:
x_text = x_text[:1000]
y = y[:1000]
train_end_idx=int(0.9 * len(y))

Jedną z użytecznych operacji przygotowania tekstu do konstrukcji klasyfikatora jest zastąpienie poszczególnych tokenów ich indeksami. Chociaż w praktyce ten proces często następuje dopiero po szeregu etapów przetwarzania tekstu takich jak tokenizacja, lematyzacja czy stemming - w tym ćwiczeniu wyodrębnimy tokeny rozdzielając tekst znakiem spacji.

Klasyfikator powinien obsługiwać także słowa, które nie występowały w zbiorze uczącym. Podstawową techniką obsługi takich słów jest wprowadzenie specjalnego tokenu UNK, obsługującego nieznane słowa. W tym celu usuwa się ze zbioru danych pewną liczbę najrzadszych słów i zastępuje się je tokenami UNK.

Zbuduj słownik `w2i` mapujący tokeny na kolejne indeksy tj. liczby naturalne. Pomiń tokeny występujące w zbiorze uczącym 5 lub mniej razy.



In [3]:
w2i = defaultdict(lambda: len(w2i))
UNK = w2i["<unk>"] #Przypisz indeks tokenowi UNK

token_counter = Counter()
for review in x_text[:train_end_idx]:
    tokens = review.split()
    token_counter.update(tokens)

for token, count in token_counter.items():
    if count > 5:
        _ = w2i[token]  # Assign a new index

n_words = len(w2i)

In [4]:
print("Vocabulary size:", n_words)
print("Index of UNK token:", UNK)

Vocabulary size: 1303
Index of UNK token: 0


Po zbudowaniu słownika `w2i`, przekonwertujmy nasz zbiór danych z listy słów na listę indeksów słów. Od razu podzielimy zbiór na część uczącą i część testową, a także przekonwertujemy klasy na indeksy klas.

In [5]:
w2i = defaultdict(lambda: UNK, w2i) # Domyślną wartością słownika jest UNK, 
         #chociaż w2i będzie zawierał wpisy do wszystkich słów to nowym tokenom będzie przypisywał indeks UNK
class2i = defaultdict(lambda: len(class2i))
        # mapuj klasy na indeksy klas
    
def read_dataset(start_idx,end_idx):
    for i, text in enumerate(x_text[start_idx:end_idx]):
        yield ([w2i[x] for x in text.split(" ")], class2i[y[i]])
        
train = list(read_dataset(0, train_end_idx))
dev = list(read_dataset(train_end_idx, len(y)))
n_class = len(class2i)

In [6]:
print(n_words, n_class)

1303 5


## Zadanie 2 - pierwszy model klasyfikacji tekstu w PyTorch
Podstawową strukturą danych w PyTorch jest tensor, na którym możesz wykonywać analogiczne operacje jak na macierzach `numpy`. Podstawową metodą stworzenia tensora jest wywołanie konstruktora `torch.tensor` na liście liczb. Istnieją także inne konstruktory tensorów, analogiczne do `numpy`. Można też na nich operować za pomocą standardowych operatorów, indeksowania, i odpowiedników innych funkcji znanych z `numpy`.



In [7]:
print(torch.tensor([1,2,3]))
print(torch.rand( (3,3) ))
print(torch.ones( (3,3) ))
print(2 * torch.ones( (3,3) ))

tensor([1, 2, 3])
tensor([[0.9433, 0.4396, 0.8386],
        [0.1933, 0.3076, 0.4182],
        [0.3057, 0.4234, 0.8319]])
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
tensor([[2., 2., 2.],
        [2., 2., 2.],
        [2., 2., 2.]])


Dlaczego więc korzystamy z PyTorch, a nie z biblioteki `numpy` skoro tensory wydają się mieć analogiczną funkcjonalność do poznanych uprzednio macierzy? Powodów jest oczywiście wiele, m.in. możliwość przeniesienia obliczeń na kartę graficzną (technologia CUDA), ale z punktu widzenia naszego ćwiczenia kluczowa jest funkcjonalność automatycznego liczenia gradientów. W przypadku konstrukcji sieci neuronowej czy modelu liniowego, w PyTorch nie jest konieczne samodzielne wyprowadzanie i implementowanie gradientu, gdyż biblioteka zrobi to za nas automatycznie.

Wyznaczanie gradientów odbywa się za pomocą algorytmu wstecznej propagacji, który ma dwie fazy: *forward* i *backward*. Faza *forward* polega na policzeniu wyniku funkcji, a faza *backward* wyznacza gradienty wszystkich jej parametrów.

W celu poznania tej funkcjonalności policzymy pochodne cząstkowe prostej funkcji kwadratowej:
$$result = x_1^2 + x_2^2+ x_3^2$$
której pochodne cząstkowe mają postać $2x_i$.

Rozpocznijmy implementacje tej funkcji od stworzenia 3-elementowego wektora zmiennych `x`.



In [8]:
x = torch.tensor([1.,2.,3.], requires_grad=True)

Jak pewnie zauważyłeś, w konstruktorze użyliśmy dodatkowego parametru `requires_grad`. Domyślnie wykonanie operacji na dowolnym tensorze nie traktuje się jako części fazy `forward`, gdyż nie do wszystkich tensorów użytych w kodzie będziemy potrzebować wartości pochodnych. Aby zasygnalizować, że dla danej zmiennej konieczne jest zapisywanie informacji o wykonywanych na niej operacjach, należy ustawić wartość jej parametru `requires_grad` na `True`.



In [9]:
print(x.requires_grad)

True


Przejdźmy do policzenia wartości wyżej zdefiniowanej funkcji.

In [10]:
result = (x**2).sum()

Tensory posiadają parametr `.grad`, który przechowuje informacje o wyznaczonym gradiencie.

In [11]:
print(x.grad)

None


W tej chwili, pomimo obliczenia wartości zmiennej `result`, wartość gradientu nie jest policzona, gdyż nie poinformowaliśmy biblioteki o zakończeniu fazy `forward` i konieczności wykonania fazy `backward`. Możemy to zrobić poprzez wykonanie funkcji `backward()` na obliczonej wartości funkcji (funkcję tę można wywołać tylko na skalarnym wyniku!).




In [12]:
result.backward()

In [13]:
x.grad

tensor([2., 4., 6.])

Zwróć uwagę, że wektor `x.grad`, zgodnie z naszymi oczekiwaniami, zawiera wartości pochodnej cząstkowej tj. `2x`. Spróbujmy jeszcze raz, licząc pochodną po logarytmie z `result`.

In [14]:
result2 = torch.log(result)

In [15]:
#result2.backward()

Niestety operacja się nie powiodła. Przed wykonaniem kolejnej fazy *backward* należy - upraszczając - wykonać fazę *forward*. Nasze poprzednie operacje konstruowały fazę *forward* od parametrów `x` aż do zmiennej z wynikiem, jednakże przy wykonaniu fazy *backward* została zwolniona pamięć przechowująca informacje o kolejno wykonywanych operacjach na tych zmiennych (graf obliczeń). Kolejna operacja została wykonana bezpośrednio na tensorze `result`, konstruując fazę *forward* od `result` do `result2`, jednak zabrakło grafu obliczeń od parametrów `x`.

Uruchomienie poniższego kodu, z operacjami rozpoczynającymi się od `x`, zakończy się obliczeniem gradientu z sukcesem.



In [16]:
result = (x**2).sum()
result2 = torch.log(result)
result2.backward()
print(x.grad)

tensor([2.1429, 4.2857, 6.4286])


Sprawdźmy poprawność uzyskanego wyniku. Zmienna $result= 1^2+2^2+3^2=14$, a pochodna z logarytmu naturalnego to $\frac{1}{x}$. W związku z tym:
$$\frac{\partial }{\partial x_1} \log result = \frac{1}{result} \cdot \frac{\partial }{\partial x_1} result = \frac{1}{result} 2x_1 $$
Przy naszych wartościach $x$ równa się to $\frac{1}{14}\cdot 2 = 0,1428$. Łatwo zauważyć, że wynik znajdujący się w tensorze `x.grad` jest błędny, a konkretnie za duży o 2 jednostki.

Stało się tak dlatego, że gradient z kolejnych faz `backward` jest akumulowany w parametrze `.grad` (poprzednia wartość policzonej pochodnej cząstkowej wynosiła właśnie 2). Takie zachowanie biblioteki może być bardzo użyteczne w sytuacji gdy chcemy w zmiennej zagregować gradienty funkcji celu liczonych na kolejno przetwarzanych instancjach lub przy treningu modelu z wieloma funkcjami celu; tutaj jednak doprowadziło to do błędnego wyniku. Z tego powodu bardzo ważne jest pamiętanie o wyzerowaniu wartości gradientów przed przystąpieniem do kolejnych obliczeń.



In [17]:
x.grad.zero_()

tensor([0., 0., 0.])

In [18]:
result = (x**2).sum()
result2 = torch.log(result)
result2.backward()
print(x.grad)

tensor([0.1429, 0.2857, 0.4286])


Zwróć uwagę na konwencję biblioteki PyTorch - jeśli nazwa funkcji zakończona jest podkreślnikiem to taka operacja jest wykonywana `in-place`. (np. `x.add(5)` - `x` nadal ma stałą wartość, `x.add_(5)` wartość `x` zwiększono o 5).

Zaimplementujmy podstawowy algorytm uczący w PyTorch. Będzie to prosta sieć neuronowa składająca się z:
- macierzy zanurzeń $C$, przetwarzającej indeksy słów na odpowiednie reprezentacje wektorowe, 
- operacji uśredniania tych zanurzeń do jednego zanurzenia (average pooling over time) 
- oraz jednej warstwy liniowej (softmax) zwracającej wynik.

Algorytmem uczącym będzie SGD optymalizujące entropię krzyżową, czyli dla kolejnych instancji uczących będziemy wykonywać:
$$parametry = parametry - \eta \nabla f\_celu$$
Implementacja ta będzie wyjątkowo prosta, gdyż gradient funkcji celu ($\nabla f\_celu$) zostanie obliczony automatycznie przez PyTorch. Ponadto entropia krzyżowa jest już zaimplementowana w PyTorch `F.cross_entropy(logits, target)`. Zwróć uwagę, że argumentem tej funkcji są wartości logitów (nie trzeba implementować funkcji softmax przetwarzającej wartości logitów na prawdopodobieństwa).

**UWAGA** W implementacji nie należy używać gotowych implementacji SGD czy warstw sieci neuronowych w PyTorch.



Pierwszym krokiem w implementacji będzie zaimplementowanie samego modelu. Należy zainicjalizować macierz $C$ przechowującą w wierszach zanurzenia dla kolejnych słów (liczba słów to `n_words`, wymiarowość zanurzenia określ na 20) oraz macierz $W$ przechowującą parametry warstwy liniowej, zwracającej wartości logitów dla każdej z klas (liczba klas to `n_class`). Macierz $W$ w dodatkowej kolumnie powinna też przechowywać wartości wyrazów wolnych (bias). Wartości zainicjalizuj losowo `torch.rand`. Pamiętaj, że dla tych macierzy będziesz potrzebował wyznaczyć potem wartości gradientów.



In [19]:
EMBEDDING_SIZE = 20
C = torch.rand((n_words, EMBEDDING_SIZE), requires_grad=True)
W = torch.rand((n_class, EMBEDDING_SIZE + 1), requires_grad=True)

In [20]:
print(C.shape)
print(W.shape)

torch.Size([1303, 20])
torch.Size([5, 21])


Zaimplementuj funkcję `simple_model`, której argumentem będzie instancja testowa (jest to więc lista indeksów słów występujących w tekście), a na której wyjściu będzie wektor `n_class`-elementowy zawierający obliczone wartości logitów.

In [21]:
def simple_model(x):
    doc_embedding = C[x].mean(0)
    doc_with_bias = torch.cat([torch.ones(1),doc_embedding]) # Skonkatenowanie 1 z uzyskaną reprezentacją (bias)
    return W @ doc_with_bias # Obliczenie wartości logitów (tj. warstwa liniowa)

Zaimplementuj algorytm SGD w poniższej pętli. Pętla ta iteruje po zbiorze uczącym oraz dla każdej instancji oblicza wartość funkcji celu. Twoje zadania:
- Policz gradienty (faza *backward*)
- Zaimplementuj aktualizacje $W$ i $C$ wg. wzoru na SGD. Operacje modyfikujące $W$ i $C$ musisz wykonać w środku klauzuli `with torch.no_grad():`, aby nie śledzić z tych operacji gradientów.
- Pamiętaj o wyczyszczeniu gradientów (zarówno w $W$ jak i $C$)



In [22]:
import torch.nn.functional as F
epochs = 5
eta = 0.5  # prędkość uczenia

for i in range(epochs):
    random.shuffle(train)
    train_loss = 0.0
    for words, tag in train:
        pred = simple_model(words)
        loss = F.cross_entropy(pred.reshape(1,-1), torch.tensor(tag).reshape(1))
        # Loss zawiera wartość funkcji celu dla przykładu, wykonaj backpropagation
        loss.backward()
        with torch.no_grad():
            C -= eta * C.grad
            W -= eta * W.grad
        C.grad.zero_()
        W.grad.zero_()
        train_loss += loss
    print("iter %r: avg. train loss=%.4f" % (i, train_loss / len(train)))

iter 0: avg. train loss=1.1018
iter 1: avg. train loss=0.9933
iter 2: avg. train loss=0.9487
iter 3: avg. train loss=0.9184
iter 4: avg. train loss=0.8952


**Ćwiczenia**
- Dlaczego w bibliotekach do głębokiego uczenia maszynowego, takich jak PyTorch, implementuje się funkcje entropii krzyżowej tak, aby przyjmowała na wejście wartości logitów zamiast prawdopodobieństw z softmax?
- Na wykładzie pokazywaliśmy warstwę zanurzeń jako warstwę mnożącą macierz $C$ przez wektor "1 z n", można ją jednak także zaimplementować jako operację odczytu odpowiedniego wiersza z macierzy. Jakie są wady i zalety obu tych sposobów implementacji?

Odpowiedź na pierwszą kropkę umieść poniżej.



- Softmax często generuje ekstremalnie małe liczby (np. 1e-30), przez co `log(softmax(x))` prowadzi do underflow i niestabilnych obliczeń.
- Łączenie softmax + log w jedną operację pozwala na stabilizację numeryczną (odejmowanie maksimum logitów przed obliczeniami).
- Obliczenia są szybsze, bo biblioteka nie musi najpierw liczyć pełnych prawdopodobieństw – gradient dla softmax + CE można zapisać w zamkniętej postaci:  

$$
\frac{\partial L}{\partial z_i} = \text{softmax}(z_i) - y_i
$$


## Zadanie 3 - wykorzystanie nn.Module


PyTorch jako biblioteka do głębokiego uczenia maszynowego oferuje nam kilka udogodnień w implementowaniu modeli uczących się, aby jeszcze bardziej uprościć ich implementację. Większość z tych udogodnień związanych z jest z reprezentowaniem modeli uczących się jako obiektów dziedziczących po `torch.nn.Module`. W obiekcie takim powinniśmy zaimplementować co najmniej konstruktor, inicjalizujący parametry modelu ($W$ i $C$), oraz funkcję `forward` obliczającą wynik modelu (we wcześniejszym zadaniu nazywaliśmy ją `simple_model`). Ponadto moduł `torch.nn` oferuje gotowe implementacje zarówno warstwy liniowej jak i warstwy zanurzeń.

Przeanalizuj poniższą implementację modelu z poprzedniego zadania.



In [23]:
class SimpleModel(torch.nn.Module):
    def __init__(self, n_words, emb_size, n_class):
        super(SimpleModel, self).__init__()
        self.embedding = torch.nn.Embedding(n_words, emb_size)
        self.linear = torch.nn.Linear(in_features=emb_size, out_features=n_class, bias=True)
        torch.nn.init.uniform_(self.embedding.weight, -0.25, 0.25)
        torch.nn.init.xavier_uniform_(self.linear.weight)

    def forward(self, words):
        emb = self.embedding(words)                 
        h = emb.mean(dim=0)                         
        h = torch.reshape(h, (1,-1))
        out = self.linear(h)              
        return out


Oprócz tego, że uzyskaliśmy elegancki obiekt reprezentujący nasz model, nie wydaje się by powyższa implementacja była krótsza czy prostsza od tej, którą uzyskaliśmy w poprzednim zadaniu bez dobrodziejstw `nn.Module`. Co zatem zyskaliśmy?

Przy implementacji modeli z dużą liczbą warstw, szczególnie uciążliwe byłoby implementowanie kolejnych linijek kodu zerujących gradienty wszystkich macierzy wag, oraz wykonywanie na nich kroków algorytmu SGD. W naszej implementacji każda macierz parametrów to dwie linijki kodu! Jednak modele dziedziczące po `torch.nn.Module` i stworzone poprzez dedykowane warstwy neuronowe posiadają gotową funkcję `parameters()` zwracającą kolejne macierze parametrów modelu.



In [24]:
model = SimpleModel(n_words, EMBEDDING_SIZE, n_class)
print([i for i in model.parameters()])

[Parameter containing:
tensor([[-0.1509, -0.1124, -0.1468,  ..., -0.0309,  0.1974, -0.1226],
        [ 0.2256, -0.0443, -0.0900,  ..., -0.0056, -0.0503,  0.1809],
        [-0.1058,  0.0936,  0.0483,  ...,  0.0934, -0.0425,  0.1871],
        ...,
        [ 0.0300,  0.0854, -0.1694,  ..., -0.1375, -0.1404, -0.0896],
        [ 0.1881, -0.0579, -0.0964,  ...,  0.0894,  0.0333,  0.0230],
        [ 0.0956,  0.0039,  0.0342,  ...,  0.1585, -0.1778, -0.1010]],
       requires_grad=True), Parameter containing:
tensor([[ 0.2341,  0.3048,  0.0142,  0.3708, -0.0373, -0.4342,  0.4601,  0.1169,
         -0.0910,  0.1528,  0.2714,  0.3013,  0.3484, -0.4828,  0.3716, -0.2158,
         -0.3344,  0.4368,  0.0021,  0.1560],
        [ 0.4578, -0.1567,  0.0327, -0.0017, -0.3423,  0.1518, -0.3797,  0.2124,
          0.2548, -0.2378,  0.2640,  0.2465, -0.4100, -0.0428,  0.4377, -0.4772,
         -0.0108,  0.3692, -0.2037,  0.0977],
        [-0.4434, -0.0355, -0.3433, -0.2272,  0.1266, -0.2091,  0.0507, -0.02

Jest to niezwykle wygodne, bo implementacja algorytmu SGD może przeiterować po tej liście parametrów i dla każdej z nich wykonać aktualizację ich wartości. Fakt, że taka lista jest tworzona automatycznie pozbawia nas ryzyka, że zwyczajnie o którejś macierzy parametrów czy wektorze wyrazów wolnych najzwyczajniej zapomnimy. 

Podobnie można zaimplementować pętlę zerującą gradienty wszystkich parametrów. Modele oferują nawet gotową taką funkcję `model.zero_grad()`, która iteruje po parametrach zerując ich gradienty. 

Zmodyfikuj implementację SGD z poprzedniego zadania, tak aby wykorzystywała `zero_grad()` i `parameters()`.



In [25]:
epochs = 5
eta = 0.5  # prędkość uczenia

for i in range(epochs):
    random.shuffle(train)
    train_loss = 0.0
    for words, tag in train:
        pred = model.forward(torch.tensor(words))
        loss = F.cross_entropy(pred, torch.tensor([tag]))
        model.zero_grad()
        loss.backward()
        with torch.no_grad():
            for param in model.parameters():
                param -= eta * param.grad
        train_loss += loss.item()
    print("iter %r: avg. train loss=%.4f" % (i, train_loss / len(train)))

iter 0: avg. train loss=0.9884
iter 1: avg. train loss=0.9574
iter 2: avg. train loss=0.9303
iter 3: avg. train loss=0.9077
iter 4: avg. train loss=0.8769


Dodatkowo moduł `torch.nn` oferuje także od razu zaimplementowane optymalizatory, w tym SGD. W konstruktorze optymalizatora należy podać listę optymalizowanych przez niego parametrów, a następnie wywołać na nim procedurę `step()` wykonującą krok algorytmu optymalizacyjnego tj. aktualizację wartości zmiennych przy użyciu gradientu. W tej sytuacji nie musisz się martwić o umieszczanie kodu zmieniającego parametry w `with torch.no_grad()` - optymalizator sam to zrobi! Optymalizator również oferuje funkcję `zero_grad()`, zerującą gradienty zmiennych wskazanych do optymalizacji.

Zmodyfikuj kod z poprzedniego zadania, tak aby wykorzystywał optymalizator SGD zaimplementowany w `torch.optim`.



In [26]:
from torch import optim
epochs = 5
eta = 0.5  # prędkość uczenia

optimizer = optim.SGD(model.parameters(), lr=eta)

for i in range(epochs):
    random.shuffle(train)
    train_loss = 0.0
    for words, tag in train:
        optimizer.zero_grad()
        pred = model(torch.tensor(words))
        loss = F.cross_entropy(pred.view(1, -1), torch.tensor([tag]))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    print("iter %r: avg. train loss=%.4f" % (i, train_loss / len(train)))

iter 0: avg. train loss=0.8562
iter 1: avg. train loss=0.8305
iter 2: avg. train loss=0.8061
iter 3: avg. train loss=0.7914
iter 4: avg. train loss=0.7520


**Ćwiczenia**
- Przeanalizuj dokładnie powyższy kod i przechodząc linia po linii, wyjaśnij co one robią z punktu widzenia treningu modelu.
- Zastanów się jak wyglądałaby Twoja własna implementacja klasy `optim.SGD`.
- Prześledź jeszcze raz implementację modelu neuronowego - pewnie w niedługim czasie będziesz implementował znacznie bardziej skomplikowane modele, tym bardziej warto je dobrze prześledzić!
- Czym różni się zaimplementowana architektura od głębokiej sieci uśredniającej?
- Na wykładzie korzystaliśmy z macierzy zanurzeń w modelach języka. Tutaj warstwa zanurzeń pojawiła się bezpośrednio w modelu klasyfikacji. Czy w uzyskanych w ten sposób zanurzeniach (zakładając dobry dobór hiperparamerów, dodanie regularyzacji itd.) zaobserwowalibyśmy podobne zależności jak te uzyskane za pomocą modelu języka? Jeśli nie, obserwacji jakich zależności między słowami spodziewałbyś się w tej reprezentacji? Skąd biorą się różnice?

Odpowiedź na ostatnią kropkę umieść poniżej.





W warstwie zanurzeń w modelu klasyfikacji słowa są reprezentowane tak, aby pomagały w przewidywaniu klasy dokumentu (np. oceny produktu). W przeciwieństwie do modeli językowych, które uczą się przewidywać kontekst słowa lub jego współwystępowanie z innymi słowami, tutaj embeddingi są optymalizowane pod kątem funkcji celu klasyfikacji.  

Dlatego w uzyskanych reprezentacjach niekoniecznie zaobserwujemy pełne semantyczne podobieństwa między słowami, jak w modelach języka (np. "krzesło" i "fotel" blisko siebie). Zamiast tego słowa o podobnym wpływie na wynik klasyfikacji będą miały podobne wektory. Na przykład wszystkie słowa bardzo pozytywnie wpływające na ocenę produktu mogą mieć zbliżone reprezentacje, niezależnie od ich znaczenia w języku.  

Różnice wynikają więc z **różnych funkcji celu**: w modelach językowych jest to przewidywanie kontekstu, a w modelu klasyfikacyjnym — przewidywanie klasy dokumentu.


# Zadanie 4
Wykorzystując wiedzę z poprzedniego zadania zaimplementuj prostą architekturę splotową do klasyfikacji tekstu i wytrenuj ją. Do jej wykonania może być przydatna klasa `torch.nn.Conv1d` i funkcja `torch.nn.ReLU` (zapoznaj się z ich dokumentacją w Internecie). Jako funkcji redukcji użyj funkcji maksimum (over time).



In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
import random

class CNN(nn.Module):
    def __init__(self, n_words, emb_size, num_filters, window_size, ntags):
        super().__init__()
        self.emb_size, self.num_filters = emb_size, num_filters
        self.embedding = nn.Embedding(n_words, emb_size)
        self.conv = nn.Conv1d(in_channels=emb_size, 
                              out_channels=num_filters, 
                              kernel_size=window_size,
                              padding=window_size-1)
        self.fc = nn.Linear(num_filters, ntags)
        torch.nn.init.xavier_uniform_(self.embedding.weight)
        torch.nn.init.xavier_uniform_(self.conv.weight)
        torch.nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, words):
        x = self.embedding(words)
        x = x.transpose(0,1).unsqueeze(0)
        x = self.conv(x)
        x = F.relu(x)
        x = torch.max(x, dim=2).values
        x = self.fc(x.view(-1))
        return x

EMBEDDING_SIZE = 20
NUM_FILTERS = 50
WINDOW_SIZE = 3
NTAGS = n_class

model = CNN(n_words, EMBEDDING_SIZE, NUM_FILTERS, WINDOW_SIZE, NTAGS)
optimizer = optim.SGD(model.parameters(), lr=0.03)

epochs = 20
for i in range(epochs):
    random.shuffle(train)
    train_loss = 0.0
    for words, tag in train:
        words_tensor = torch.tensor(words)
        target = torch.tensor([tag])
        
        optimizer.zero_grad()
        pred = model(words_tensor)
        loss = F.cross_entropy(pred.view(1,-1), target)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    print(f"iter {i}: avg. train loss = {train_loss / len(train):.4f}")


iter 0: avg. train loss = 0.9551
iter 1: avg. train loss = 0.9210
iter 2: avg. train loss = 0.9074
iter 3: avg. train loss = 0.8957
iter 4: avg. train loss = 0.8734
iter 5: avg. train loss = 0.8493
iter 6: avg. train loss = 0.8217
iter 7: avg. train loss = 0.7762
iter 8: avg. train loss = 0.7262
iter 9: avg. train loss = 0.6488
iter 10: avg. train loss = 0.5714
iter 11: avg. train loss = 0.4658
iter 12: avg. train loss = 0.3795
iter 13: avg. train loss = 0.2597
iter 14: avg. train loss = 0.1927
iter 15: avg. train loss = 0.1274
iter 16: avg. train loss = 0.0756
iter 17: avg. train loss = 0.0520
iter 18: avg. train loss = 0.0357
iter 19: avg. train loss = 0.0258
